# ERC : flux d'entrée, suivi hebdomadaire estimé contre pesé

Le modèle de ligne de démontage et de tri part de ce flux : il n'est fiable qu'à la granularité où l'estimation (SQL × poids unitaires) est confirmée par les pesées.

| # | Contenu |
|---|---|
| 1 | Composants avec leur type de déchet et leur benne |
| 2 | Masse estimée par semaine et par flux, avec marge d'erreur |
| 3 | Rapprochement avec les pesées terrain : écart, dans la marge ou non, causes candidates |
| 4 | Verdict par flux : utilisable par le modèle ou donnée à corriger d'abord |

À exécuter après `ERC_Aide_Decision_Composants.ipynb`, dans le même dossier. Pesées : `Acquisition/pesees_hebdo.csv` (séparateur `;`), colonnes `Semaine;Flux;Masse mesurée (kg)` et, facultatives, `Incertitude mesure (kg);Périmètre;Commentaire`. `Flux` doit reprendre le nom de la benne (ou du type de déchet, selon `PAR`).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJET = Path.cwd()
sys.path.insert(0, str(PROJET))
import erc_flux_hebdo as efh

F_AIDE = PROJET / "ERC_Aide_Decision_Composants.xlsx"
F_PESEES = PROJET / "Acquisition" / "pesees_hebdo.csv"
F_SORTIE = PROJET / "ERC_Flux_Entree_Suivi_Hebdo.xlsx"
PAR = "Benne"                      # ou "Type de déchet"
efh.INCERTITUDE_POIDS = None       # ex. 0.10 quand la dispersion des poids unitaires est mesurée
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 220)

cl = pd.read_excel(F_AIDE, sheet_name="Classement_Tous_ateliers")
ev, couverts = efh.charger_evenements(PROJET)
print(f"{len(ev)} lignes de consommation, {ev['Qte'].sum():.0f} pièces, {len(couverts)} jours extraits")

## 1. Composants et type de déchet

In [ ]:
comp = efh.tableau_composants(cl)
synthese = comp.groupby(["Benne", "Type de déchet"], dropna=False).agg(
    Références=("Référence", "nunique"), Pièces=("Pièces", "sum"), **{"Références au poids connu": ("Poids connu", "sum"),
    "Kg (poids connus)": ("Kg (poids connus)", "sum")}).reset_index()
display(synthese)
comp.head(30)

## 2. Masse estimée par semaine

Borne basse : pièces au poids connu seulement. Borne haute : les pièces sans poids supposées au poids moyen de leur flux. La marge d'erreur est la demi-largeur de cet intervalle ; elle se resserre à mesure que les poids manquants sont pesés.

In [ ]:
est = efh.estimation_hebdo(ev, comp, couverts, par=PAR)
est.round(2)

## 3. Estimé contre pesé

Seules les semaines qui ont une pesée sont comparées. « Dans la marge » tient compte de l'incertitude de la balance. Le cumul par flux absorbe le décalage des bacs individuels : si l'écart cumulé est nettement plus faible que l'écart de la semaine, c'est un décalage dans le temps et non une erreur de niveau. La dernière colonne est à remplir avec la cause constatée sur place.

In [ ]:
if F_PESEES.is_file():
    pesees = pd.read_csv(F_PESEES, sep=";", encoding="utf-8-sig", decimal=",")
    rap = efh.rapprocher(est, pesees)
else:
    print("Pas encore de fichier de pesées :", F_PESEES)
    rap = efh.rapprocher(est, pd.DataFrame(columns=["Semaine", "Flux", "Masse mesurée (kg)"]))
vue = ["Semaine", "Flux", "Jours extraits", "Estimé bas (kg)", "Estimé central (kg)", "Estimé haut (kg)", "Marge d'erreur (± kg)", "Masse mesurée (kg)",
       "Écart (kg)", "Écart (%)", "Dans la marge", "Écart cumulé (%)", "Causes candidates de l'écart"]
rap[vue].round(2)

## 4. Verdict par flux

In [ ]:
comparables = rap[rap["Dans la marge"].ne("non comparable")]
verdict = comparables.groupby("Flux").agg(**{"Semaines comparées": ("Semaine", "nunique"), "Dans la marge": ("Dans la marge", lambda s: (s == "oui").sum()),
                                           "Écart moyen (%)": ("Écart (%)", "mean"), "Écart cumulé final (%)": ("Écart cumulé (%)", "last"),
                                           "Couverture poids moyenne": ("Couverture poids", "mean")}).reset_index()
verdict["Utilisable par le modèle"] = np.where((verdict["Semaines comparées"] >= 3) & (verdict["Écart cumulé final (%)"].abs() <= 0.15), "oui",
                                              np.where(verdict["Semaines comparées"] < 3, "pas assez de semaines (3 minimum)", "non : corriger la cause dominante"))
display(verdict.round(3))
with pd.ExcelWriter(F_SORTIE) as w:
    for n, d in {"Composants_type_dechet": comp, "Synthese_bennes": synthese, "Estime_hebdo": est, "Rapprochement": rap, "Verdict": verdict}.items():
        d.to_excel(w, sheet_name=n, index=False)
print("Écrit :", F_SORTIE.name)